# 03. Definição de Target e Merge Macro-Micro

Este notebook é responsável por aplicar a lógica de negócio aos dados longitudinais. Os principais objetivos são:
1. **Definição do Evento de Stress (Target):** Identificar os anos em que as empresas entraram em dificuldade financeira.
2. **Mapeamento da Recorrência (Episódios):** Criar identificadores temporais para modelação de sobrevivência (tempo até ao evento e recaídas).
3. **Merge com Dados Macro:** Incorporar a conjuntura económica do país no exato ano de operação da empresa.
4. **Exportação:** Gerar o dataset final pronto para *split* e modelação (pasta `processed`).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Carregamento dos Dados Interim

In [2]:
df_micro = pd.read_csv('../data/interim/micro_long.csv')
df_macro = pd.read_csv('../data/interim/macro_consolidated.csv')

print(f"Microdata shape: {df_micro.shape}")
print(f"Macrodata shape: {df_macro.shape}")

Microdata shape: (127621, 30)
Macrodata shape: (21, 11)


## 2. Definição do Evento de Stress Financeiro (Target)

Baseado na literatura (Borges & Carvalho 2024; Fernandes 2024), uma empresa está em stress (`Target = 1`) no ano $t$ se verificar um dos seguintes critérios operacionais ou legais:

1. **Insolvência Técnica:** Capital Próprio (`Equity`) < 0.
2. **Incapacidade Operacional de Cobertura:** `EBITDA` < `Interests` (Gastos de Financiamento).
3. **Status Legal:** `Status` indica encerramento, liquidação ou falência.

In [3]:
def safe_float(col):
    return pd.to_numeric(col.astype(str).str.replace(',', '.').str.replace('[^0-9.-]', '', regex=True), errors='coerce')

# Convert critical columns to numeric
df_micro['Equity_num'] = safe_float(df_micro['Equity'])
df_micro['EBITDA_num'] = safe_float(df_micro['EBITDA'])
df_micro['Interests_num'] = safe_float(df_micro['Interests'])

# Criterion 1: Negative equity
c1_equity = df_micro['Equity_num'] < 0

# Criterion 2: EBITDA < Interest expenses (only when interest > 0)
c2_ebitda_interest = (df_micro['EBITDA_num'] < df_micro['Interests_num']) & (df_micro['Interests_num'] > 0)

# Criterion 3: Terminal status (based on common closure keywords)
terminal_keywords = ['Dissolução', 'Liquidação', 'Falência', 'Insolvente', 'Encerrada']
c3_status = df_micro['Status'].fillna('').str.contains('|'.join(terminal_keywords), case=False, na=False)

# Target assignment
df_micro['Target'] = np.where(c1_equity | c2_ebitda_interest | c3_status, 1, 0)

print("Distribuição do Target (Anos-Empresa):")
print(df_micro['Target'].value_counts(normalize=True) * 100)

Distribuição do Target (Anos-Empresa):
Target
0    62.191175
1    37.808825
Name: proportion, dtype: float64


## 3. Mapeamento da Recorrência (Episódios)

Na análise de sobrevivência recorrente, precisamos saber há quantos anos a empresa está 'em risco' no episódio atual. 
Ordenamos os dados por NIF e Ano, e criamos um contador de tempo e de episódios.

In [4]:
df_micro = df_micro.sort_values(by=['NIF Code', 'Year']).reset_index(drop=True)

# Identify Target state transitions (0 -> 1 or 1 -> 0)
# A state change marks the end of one episode and the start of another.
df_micro['State_Change'] = df_micro.groupby('NIF Code')['Target'].diff().ne(0).astype(int)

# The first year for each firm is not a real state change, force it to 0
df_micro.loc[df_micro.groupby('NIF Code').head(1).index, 'State_Change'] = 0

# Episode id is the cumulative sum of state changes
df_micro['Episode'] = df_micro.groupby('NIF Code')['State_Change'].cumsum() + 1

# Time within current episode (duration)
df_micro['Time_in_Episode'] = df_micro.groupby(['NIF Code', 'Episode']).cumcount() + 1

df_micro.drop(columns=['State_Change'], inplace=True)

## 3.5 Tratamento de Truncatura à Esquerda (Left Truncation) e Idade

Como os dados Macroeconómicos começam em 2005, o nosso período de observação efetivo inicia-se nesse ano. Empresas criadas antes de 2005 entram no estudo com 'Entrada Tardia' (Delayed Entry). Para isso, definimos a escala de tempo não como o ano de calendário, mas como a **Idade da Empresa**, criando as variáveis `t_start` e `t_stop` para o formato *Counting Process*.

In [5]:
# Filter observation period to ensure macro data coverage
df_micro = df_micro[df_micro['Year'] >= 2005].copy()

# Extract firm founding year
df_micro['Date of Establishment'] = pd.to_datetime(df_micro['Date of Establishment'], errors='coerce')
df_micro['Establishment_Year'] = df_micro['Date of Establishment'].dt.year

# Drop rows where founding year is missing (firm age cannot be calculated)
df_micro = df_micro.dropna(subset=['Establishment_Year'])

# Calculate t_stop and t_start based on firm age
df_micro['t_stop'] = df_micro['Year'] - df_micro['Establishment_Year']
df_micro['t_start'] = df_micro['t_stop'] - 1

# Remove rows with negative age (data errors)
df_micro = df_micro[df_micro['t_stop'] > 0]

## 4. Merge Macro-Micro

Juntamos os indicadores macroeconómicos alinhando a coluna `Year` do micro com a coluna `Ano` do macro.

In [6]:
# Ensure temporal key columns are the same type (integer)
df_micro['Year'] = df_micro['Year'].astype(int)
df_macro['year'] = df_macro['year'].astype(int)

# Left merge: keep all micro rows and attach corresponding macro information
df_final = pd.merge(
    df_micro, 
    df_macro, 
    left_on='Year', 
    right_on='year', 
    how='left'
)

if 'year' in df_final.columns:
    df_final.drop(columns=['year'], inplace=True)

print(f"Shape final após merge: {df_final.shape}")

Shape final após merge: (109526, 49)


## 5. Exportação para Processed

In [7]:
# Drop temporary numeric columns (they replace original string columns)
df_final.drop(columns=['Equity', 'EBITDA', 'Interests'], inplace=True)
df_final.rename(columns={'Equity_num': 'Equity', 'EBITDA_num': 'EBITDA', 'Interests_num': 'Interests'}, inplace=True)

df_final.to_csv('../data/processed/final_survival_dataset.csv', index=False)
print("Dataset exportado com sucesso para 'data/processed/final_survival_dataset.csv'")

Dataset exportado com sucesso para 'data/processed/final_survival_dataset.csv'
